In [ ]:
!git clone https://github.com/Ingaiza/CRNN.git


Cloning into 'CRNN'...
remote: Enumerating objects: 227, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 227 (delta 46), reused 62 (delta 31), pack-reused 133 (from 1)
Receiving objects: 100% (227/227), 13.74 MiB | 39.31 MiB/s, done.
Resolving deltas: 100% (114/114), done.


In [ ]:
!pip install SoundFile
!pip install pysoundfile
!pip install tensorboardX
!pip install xgboost pandas
!pip install git+https://github.com/ksanjeevan/torchparse.git


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.3 MB/s eta 0:00:00
  Cloning https://github.com/ksanjeevan/torchparse.git to /tmp/pip-req-build-7nvtq5tm
  Running command git clone --filter=blob:none --quiet https://github.com/ksanjeevan/torchparse.git /tmp/pip-req-build-7nvtq5tm
  Resolved https://github.com/ksanjeevan/torchparse.git to commit cf4fe054e657eaa86406a41c8dd06f68fc01e40d
  Preparing metadata (setup.py) ... done
  Created wheel for torchparse: filename=torchparse-0.1-py3-none-any.whl size=7970 sha256=b7b5ffcaccd0c42021d96c6930e2904f6b8273280c9b06cc9bcd437d745b7769
  Stored in directory: /tmp/pip-ephem-wheel-cache-iru0b8bs/wheels/73/e5/50/87fec598c8d8948cf7d80993470aff818d0977afdd6bdcf3b7
Successfully built torchparse


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WATCH_FOLDER = "/content/drive/MyDrive/Ambience"
PROCESSED_FOLDER = "/content/drive/MyDrive/Processed"



Mounted at /content/drive


In [ ]:
%cd CRNN/
!git checkout mixed
%ls

/content/CRNN
Branch 'mixed' set up to track remote branch 'mixed' from 'origin'.
Switched to a new branch 'mixed'
colab.py     eval/                 hybrid_short.py   net/           run.py*
config.json  extract_and_train.py  LICENSE           predict.py     train/
crnn.cfg     hybrid_both.py        mix_and_train.py  README.md      utils/
data/        hybrid.py             models/           result_plots/  web.html


In [ ]:
import sys
import os

REPO_PATH = "/content/CRNN"
sys.path.append(REPO_PATH)

try:
    from net.model import AudioCRNN
    print("SUCCESS: Repository code is accessible!")
except ImportError:
    print("ERROR: Could not find 'net.model'. Make sure REPO_PATH points to the cloned folder.")

SUCCESS: Repository code is accessible!


In [ ]:
import os
import time
import glob
import shutil
import requests
import torch
import soundfile as sf
import numpy as np
import xgboost as xgb
import itertools
import torchaudio.transforms as T
import torch.nn.functional as F
import random
import sys
from unittest.mock import MagicMock

# --- 1. MOCKING & ALIASING ---
try:
    import utils
    sys.modules["utils"] = utils
    sys.modules["utils.logger"] = utils.logger
    sys.modules["utils.util"] = utils.util
except ImportError:
    utils_mock = MagicMock()
    utils_mock.logger = MagicMock()
    class MockLogger:
        def __init__(self, *args, **kwargs): pass
    utils_mock.logger.Logger = MockLogger
    sys.modules["utils"] = utils_mock
    sys.modules["utils.logger"] = utils_mock.logger

from net.model import AudioCRNN

# CONFIGURATION
CRNN_MODEL_PATH = "/content/CRNN/models/model_best.pth"
XGB_MODEL_PATH = "/content/CRNN/models/xgboost_mixed_model.json"
CFG_PATH = "/content/CRNN/crnn.cfg"

# FOLDERS
WATCH_FOLDER = "/content/drive/MyDrive/Ambience"
PROCESSED_FOLDER = "/content/drive/MyDrive/Forest_Audio/Processed"

WEBHOOK_URL = "https://redressible-nonviable-tanja.ngrok-free.dev/alert"

CLASS_MAP = {0: "Natural", 1: "Unnatural", 2: "Human Sound"}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# NODE LOCATIONS (Simulation)
NODE_LOCATIONS = [
    {"id": 0, "lat": 0.3520, "lng": 34.8650, "name": "Buyangu Hill (North)"},      # High visibility point
    {"id": 1, "lat": 0.3850, "lng": 34.8950, "name": "Kisere Fragment"},           # Detached forest section (High Risk)
    {"id": 2, "lat": 0.3150, "lng": 34.8450, "name": "Isiukhu River Crossing"},    # Mining threat zone
    {"id": 3, "lat": 0.3280, "lng": 34.8200, "name": "Salazar Circuit"},           # Logging threat
    {"id": 4, "lat": 0.2820, "lng": 34.8640, "name": "Isecheno Station"},          # Central Research Hub
    {"id": 5, "lat": 0.2650, "lng": 34.8420, "name": "Lirhanda Hill"},             # Strategic overlook
    {"id": 6, "lat": 0.2300, "lng": 34.8850, "name": "Yala River Border"},         # Water access / Mining
    {"id": 7, "lat": 0.2150, "lng": 34.8550, "name": "Kibiri Block (South)"},      # Dense population border
    {"id": 8, "lat": 0.3600, "lng": 34.8100, "name": "Malava Edge"},               # Agriculture encroachment
    {"id": 9, "lat": 0.2600, "lng": 34.9100, "name": "Ikuywa River East"},         # Eastern access route
    {"id": 10, "lat": 0.2900, "lng": 34.8200, "name": "Pump House Sector"},        # Infrastructure protection
    {"id": 11, "lat": 0.1900, "lng": 34.8400, "name": "Kaimosi Border"},           # Southern-most tip
    {"id": 12, "lat": 0.2800, "lng": 34.7900, "name": "Mukumu West Gate"},         # Western community access
    {"id": 13, "lat": 0.3100, "lng": 34.9200, "name": "Cheenya Edge"},             # Eastern boundary
    {"id": 14, "lat": 0.3000, "lng": 34.8500, "name": "Colobus Trail Inner"},      # Deep forest primate habitat
]

# VALIDATION LAYER
def validate_prediction(probs):
    original_winner_idx = np.argmax(probs)
    w_low = 0.76
    w_high = 1.24
    multipliers = [w_low, w_high]
    combinations = list(itertools.product(multipliers, repeat=3))
    wins = 0
    total_scenarios = len(combinations)

    for coeffs in combinations:
        weighted_probs = np.array(probs) * np.array(coeffs)
        round_winner_idx = np.argmax(weighted_probs)
        if round_winner_idx == original_winner_idx:
            wins += 1

    is_valid = wins > (total_scenarios / 2)
    pass_ratio = wins / total_scenarios
    return is_valid, pass_ratio

# INITIALIZATION
if not os.path.exists(WATCH_FOLDER): os.makedirs(WATCH_FOLDER)
if not os.path.exists(PROCESSED_FOLDER): os.makedirs(PROCESSED_FOLDER)

print("Loading Models...")
config = {'cfg': CFG_PATH, 'transforms': {'args': {'channels': 'mono'}}}
crnn = AudioCRNN(classes=CLASS_MAP.values(), config=config)
checkpoint = torch.load(CRNN_MODEL_PATH, map_location=DEVICE, weights_only=False)
if isinstance(checkpoint, dict):
    state_dict = checkpoint.get('model', checkpoint.get('state_dict', checkpoint))
    crnn.load_state_dict(state_dict)
else:
    crnn.load_state_dict(checkpoint)
crnn.eval().to(DEVICE)

xgb_model = xgb.XGBClassifier()
xgb_model.load_model(XGB_MODEL_PATH)
print(f"Models Loaded. Watching '{WATCH_FOLDER}'...")

# MAIN LOOP
while True:
    wav_files = glob.glob(os.path.join(WATCH_FOLDER, "*.wav"))

    if not wav_files:
        time.sleep(1)
        continue

    wav_files.sort(key=os.path.getmtime)
    current_file = wav_files[0]
    filename = os.path.basename(current_file)

    print(f"\nProcessing: {filename}")

    try:
        # Loading Audio
        data, sr = sf.read(current_file)
        waveform = torch.from_numpy(data).float()

        if waveform.ndim == 1: waveform = waveform.unsqueeze(0)
        else: waveform = waveform.permute(1, 0)

        if sr != 16000:
            resampler = T.Resample(sr, 16000)
            waveform = resampler(waveform)

        if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)

        seqs = waveform.permute(1, 0).unsqueeze(0)
        lengths = torch.tensor([seqs.shape[1]]).long()
        srs = torch.tensor([16000]).long()
        batch = (seqs.to(DEVICE), lengths.to(DEVICE), srs.to(DEVICE))

        # Hybrid Inference
        with torch.no_grad():
            features = crnn(batch, return_features=True)
            features_np = features.cpu().numpy()
            probs = xgb_model.predict_proba(features_np)[0]

        # Validation Layer
        is_valid, pass_ratio = validate_prediction(probs)
        pred_idx = np.argmax(probs)
        pred_class = CLASS_MAP[pred_idx]

        # Randomizong Location (Simulation)
        current_node = random.choice(NODE_LOCATIONS)

        # Getting Fire Status from Filename
        # "Ambience_YES_..." or "Ambience_NO_..."
        is_fire = False
        if "_YES_" in filename:
            is_fire = True
        elif "_NO_" in filename:
            is_fire = False

        # Print Debug Info
        print(f"-> Location: {current_node['lat']}, {current_node['lng']}")
        print(f"-> Fire Detected in Filename? {is_fire}")
        print(f"-> Audio Prediction: {pred_class} ({probs[pred_idx]:.2%})")
        print(f"-> Validation: Won {pass_ratio:.0%} of scenarios.")

        if is_valid:
            print("-> STATUS: VALID ALERT. Sending to UI...")
            payload = {
                "class": pred_class,
                "probs": [float(p) for p in probs],
                "filename": filename,
                "lat": current_node["lat"],
                "lng": current_node["lng"],
                "fire": is_fire
            }
            try:
                requests.post(WEBHOOK_URL, json=payload, timeout=2)
                print("   Success: Webhook sent.")
            except Exception as e:
                print(f"   Error sending webhook: {e}")
        else:
            print(f"-> STATUS: BLOCKED. Prediction '{pred_class}' too weak/noisy.")

        # Cleanup
        shutil.move(current_file, os.path.join(PROCESSED_FOLDER, filename))

    except Exception as e:
        print(f"CRITICAL ERROR processing {filename}: {e}")
        error_folder = os.path.join(PROCESSED_FOLDER, "Errors")
        if not os.path.exists(error_folder): os.makedirs(error_folder)
        shutil.move(current_file, os.path.join(error_folder, filename))

Loading Models...
Models Loaded. Watching '/content/drive/MyDrive/Ambience'...

Processing: Ambience_YES_20251120_153107.wav
-> Location: 0.315, 34.845
-> Fire Detected in Filename? True
-> Audio Prediction: Human Sound (43.97%)
-> Validation: Won 75% of scenarios.
-> STATUS: VALID ALERT. Sending to UI...
   Success: Webhook sent.

Processing: Ambience_NO_20251120_152701.wav
-> Location: 0.282, 34.864
-> Fire Detected in Filename? False
-> Audio Prediction: Human Sound (62.62%)
-> Validation: Won 100% of scenarios.
-> STATUS: VALID ALERT. Sending to UI...
   Success: Webhook sent.

Processing: Ambience_NO_20251120_152805.wav
-> Location: 0.29, 34.82
-> Fire Detected in Filename? False
-> Audio Prediction: Unnatural (57.32%)
-> Validation: Won 100% of scenarios.
-> STATUS: VALID ALERT. Sending to UI...
   Success: Webhook sent.

Processing: Ambience_NO_20251120_152137.wav
-> Location: 0.328, 34.82
-> Fire Detected in Filename? False
-> Audio Prediction: Unnatural (48.60%)
-> Validation: